In [ ]:
!pip install -q gradio diffusers transformers accelerate peft "torchao>=0.16.0"

import gradio as gr
import torch
import gc
import traceback
from diffusers import StableDiffusionPipeline
from transformers import pipeline

print("Carregando modelos na Placa de Vídeo...\n")

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", 
    torch_dtype=torch.float16
).to("cuda")
pipe.load_lora_weights("LuisCurado/lora-estilo-rupestre")

expansor = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device="cuda")
tts = pipeline("text-to-speech", model="suno/bark-small", device="cuda")

def gerar_multimodal(tema):
    print(f"\n--- NOVA TENTATIVA: {tema} ---")
    try:
        print("1. Iniciando expansão de texto...")
        instrucao = f"Expanda o tema '{tema}' em uma descricao visual rica, em uma frase, iniciando com 'estilo_rupestre, '"
        # O [0]["generated_text"] pode estar falhando se o modelo retornar em outro formato
        prompt_expandido = expansor(instrucao, max_new_tokens=60)[0]["generated_text"]
        print(f"-> Sucesso LLM! Texto gerado: {prompt_expandido}")
        
        torch.cuda.empty_cache()
        gc.collect()
        
        print("2. Iniciando geração da imagem (Isso pode demorar)...")
        imagem = pipe(
            prompt_expandido, 
            negative_prompt="desfocado, deformado, moderno", 
            guidance_scale=7.5, 
            num_inference_steps=30
        ).images[0]
        print("-> Sucesso Imagem!")
        
        torch.cuda.empty_cache()
        gc.collect()
        
        print("3. Iniciando geração de áudio (Isso pode demorar)...")
        audio = tts(prompt_expandido)
        print("-> Sucesso Áudio!")
        
        print("--- FINALIZADO COM SUCESSO ---")                
        return prompt_expandido, imagem, (audio["sampling_rate"], audio["audio"].squeeze())
    
    except Exception as e:
        print("\n🚨 ERRO CAPTURADO NO BACKEND 🚨")
        # Isso força o erro completo a aparecer na sua tela do Colab!
        traceback.print_exc() 
        raise gr.Error(f"Erro capturado. Olhe o terminal do Colab!")

demo = gr.Interface(
    fn=gerar_multimodal, 
    inputs=gr.Textbox(label="Tema (Ex: feira de domingo)"),
    outputs=[
        gr.Textbox(label="Prompt expandido pelo LLM"), 
        gr.Image(label="Arte Rupestre Gerada"), 
        gr.Audio(label="Narração da Cena")
    ]
)

# O debug=True prende o terminal para garantir que todo log seja exibido
demo.launch(share=True, debug=True)